In [12]:
import pandas as pd
import numpy as np

df = pd.read_csv("DataCoSupplyChainDataset.csv",
    encoding="cp1252"
)

In [13]:
print(df.shape)

(180519, 53)


In [14]:
print(df.columns.tolist())

['Type', 'Days for shipping (real)', 'Days for shipment (scheduled)', 'Benefit per order', 'Sales per customer', 'Delivery Status', 'Late_delivery_risk', 'Category Id', 'Category Name', 'Customer City', 'Customer Country', 'Customer Email', 'Customer Fname', 'Customer Id', 'Customer Lname', 'Customer Password', 'Customer Segment', 'Customer State', 'Customer Street', 'Customer Zipcode', 'Department Id', 'Department Name', 'Latitude', 'Longitude', 'Market', 'Order City', 'Order Country', 'Order Customer Id', 'order date (DateOrders)', 'Order Id', 'Order Item Cardprod Id', 'Order Item Discount', 'Order Item Discount Rate', 'Order Item Id', 'Order Item Product Price', 'Order Item Profit Ratio', 'Order Item Quantity', 'Sales', 'Order Item Total', 'Order Profit Per Order', 'Order Region', 'Order State', 'Order Status', 'Order Zipcode', 'Product Card Id', 'Product Category Id', 'Product Description', 'Product Image', 'Product Name', 'Product Price', 'Product Status', 'shipping date (DateOrde

In [15]:
cols = [
    "order date (DateOrders)",
    "shipping date (DateOrders)",
    "Days for shipping (real)",
    "Days for shipment (scheduled)",
    "Shipping Mode",
    "Order Region"
]

print(df[cols].head())
print(df[cols].isnull().sum())

  order date (DateOrders) shipping date (DateOrders)  \
0         1/31/2018 22:56             2/3/2018 22:56   
1         1/13/2018 12:27            1/18/2018 12:27   
2         1/13/2018 12:06            1/17/2018 12:06   
3         1/13/2018 11:45            1/16/2018 11:45   
4         1/13/2018 11:24            1/15/2018 11:24   

   Days for shipping (real)  Days for shipment (scheduled)   Shipping Mode  \
0                         3                              4  Standard Class   
1                         5                              4  Standard Class   
2                         4                              4  Standard Class   
3                         3                              4  Standard Class   
4                         2                              4  Standard Class   

     Order Region  
0  Southeast Asia  
1      South Asia  
2      South Asia  
3         Oceania  
4         Oceania  
order date (DateOrders)          0
shipping date (DateOrders)       0
Days

In [16]:
df["order date (DateOrders)"] = pd.to_datetime(
    df["order date (DateOrders)"], errors="coerce"
)

df["shipping date (DateOrders)"] = pd.to_datetime(
    df["shipping date (DateOrders)"], errors="coerce"
)

df["Processing_Duration"] = (
    df["shipping date (DateOrders)"]
    - df["order date (DateOrders)"]
).dt.total_seconds() / 86400

print(df[["Processing_Duration"]].describe())

       Processing_Duration
count        180519.000000
mean              3.498826
std               1.617394
min               0.500000
25%               2.000000
50%               3.000000
75%               5.000000
max               6.000000


In [17]:
df["Transit_Duration_Proxy"] = pd.to_numeric(
    df["Days for shipping (real)"],
    errors="coerce"
)

print(df[
    ["Processing_Duration", "Transit_Duration_Proxy"]
].describe())

       Processing_Duration  Transit_Duration_Proxy
count        180519.000000           180519.000000
mean              3.498826                3.497654
std               1.617394                1.623722
min               0.500000                0.000000
25%               2.000000                2.000000
50%               3.000000                3.000000
75%               5.000000                5.000000
max               6.000000                6.000000


In [18]:
mode_comparison = (
    df.groupby("Shipping Mode")
    .agg(
        Total_Orders=("Order Id", "nunique"),
        Avg_Processing_Days=("Processing_Duration", "mean"),
        Avg_Transit_Proxy_Days=("Transit_Duration_Proxy", "mean")
    )
    .round(2)
    .reset_index()
)

print(mode_comparison)

    Shipping Mode  Total_Orders  Avg_Processing_Days  Avg_Transit_Proxy_Days
0     First Class         10079                 2.00                    2.00
1        Same Day          3571                 0.50                    0.48
2    Second Class         12778                 3.99                    3.99
3  Standard Class         39324                 4.00                    4.00


In [19]:
region_comparison = (
    df.groupby("Order Region")
    .agg(
        Total_Orders=("Order Id", "nunique"),
        Avg_Processing_Days=("Processing_Duration", "mean"),
        Avg_Transit_Proxy_Days=("Transit_Duration_Proxy", "mean")
    )
    .round(2)
    .reset_index()
    .sort_values("Avg_Processing_Days", ascending=False)
)

print(region_comparison)

       Order Region  Total_Orders  Avg_Processing_Days  Avg_Transit_Proxy_Days
2    Central Africa           556                 3.57                    3.56
19      West Africa          1223                 3.54                    3.53
7      Eastern Asia          3318                 3.53                    3.53
5       East Africa           613                 3.52                    3.51
3   Central America          9396                 3.51                    3.51
21     West of USA           2667                 3.51                    3.50
8    Eastern Europe          1292                 3.51                    3.50
10  Northern Europe          3716                 3.51                    3.50
1         Caribbean          2806                 3.51                    3.51
11          Oceania          4362                 3.50                    3.50
13       South Asia          3335                 3.50                    3.50
15   Southeast Asia          4356                 3.

In [20]:
region_mode_comparison = (
    df.groupby(["Order Region", "Shipping Mode"])
    .agg(
        Total_Orders=("Order Id", "nunique"),
        Avg_Processing_Days=("Processing_Duration", "mean"),
        Avg_Transit_Proxy_Days=("Transit_Duration_Proxy", "mean")
    )
    .round(2)
    .reset_index()
)

print(region_mode_comparison.head(20))

       Order Region   Shipping Mode  Total_Orders  Avg_Processing_Days  \
0            Canada     First Class            58                 2.00   
1            Canada        Same Day            12                 0.50   
2            Canada    Second Class            55                 4.03   
3            Canada  Standard Class           184                 3.73   
4         Caribbean     First Class           415                 2.00   
5         Caribbean        Same Day           155                 0.50   
6         Caribbean    Second Class           541                 4.03   
7         Caribbean  Standard Class          1695                 3.96   
8    Central Africa     First Class            96                 2.00   
9    Central Africa        Same Day            29                 0.50   
10   Central Africa    Second Class           100                 4.13   
11   Central Africa  Standard Class           331                 4.12   
12  Central America     First Class   

In [21]:
output_path = (
    r"C:\Users\HP\Internship\07_Processing_Transit_Comparison.xlsx"
)

with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
    mode_comparison.to_excel(
        writer, sheet_name="Shipping_Mode", index=False
    )

    region_comparison.to_excel(
        writer, sheet_name="Region_Comparison", index=False
    )

    region_mode_comparison.to_excel(
        writer, sheet_name="Region_Mode", index=False
    )

print("Excel file created successfully!")

Excel file created successfully!
